# Tutorial 3: What a Checkpoint Holds, and What It Doesn't

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/tutorials/03-train-tiny-model.ipynb)

**Reading. About 15 minutes; it runs in under a minute.**

A saved model is a file of numbers. Each row of the model's first layer belongs to one word, but
*which* word is not in the file: that is the vocabulary's job, and for a word-level model like
A8's the vocabulary is rebuilt from the training data, not saved. Rebuild it a little
differently and the numbers land on the wrong words. Sometimes that fails loudly; sometimes the
model loads without complaint and translates nonsense, and nothing tells you.

By the end you will be able to:

1. say what a TorchLingo checkpoint file contains, and what it leaves out;
2. rebuild a saved model so that it is the model you trained;
3. recognise the two ways a rebuild goes wrong: a size mismatch that stops you, and a reordered vocabulary that does not.

These serve Lecture 8a's outcome *start, checkpoint and resume a long run*, and the A8
notebook's reload step depends on them.

**Running in Google Colab?** Run the next two cells: the first installs TorchLingo, the second
checks the setup. There is nothing to uncomment, and no GPU is needed.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**TorchLingo tutorial** — assigned reading for Lecture 7 (*Research Paper Reviews; Intro to Neural Networks*). A head start on **A8**.

In [1]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo

In [2]:
# Check the setup: the TorchLingo and PyTorch versions and the device, printed here so that
# anything wrong shows up now rather than ten cells from now.
from torchlingo.colab import setup

env = setup()

TorchLingo 0.2.6 (editable checkout) | PyTorch 2.13.0 | device cpu | not in Colab


In [3]:
import random
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

from torchlingo.checkpoint import load_checkpoint, save_checkpoint
from torchlingo.config import Config
from torchlingo.data_processing import NMTDataset, collate_fn
from torchlingo.inference import translate_batch
from torchlingo.models import SimpleTransformer
from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer

device = env.device

## Part 1: Train a model worth saving

Sixteen short phrases, each repeated twelve times, so a tiny model learns them all in a few
seconds and translates them correctly. That matters here: a broken rebuild is only visible
against a model that was right to begin with.

In [4]:
PAIRS = [
    ("a red car", "un coche rojo"), ("a blue car", "un coche azul"),
    ("a green door", "una puerta verde"), ("a white door", "una puerta blanca"),
    ("the old house", "la casa vieja"), ("the new house", "la casa nueva"),
    ("the big window", "la ventana grande"), ("the small window", "la ventana pequeña"),
    ("two black cats", "dos gatos negros"), ("three yellow birds", "tres pájaros amarillos"),
    ("my brother sings", "mi hermano canta"), ("my sister reads", "mi hermana lee"),
    ("we eat bread", "comemos pan"), ("they drink water", "beben agua"),
    ("it is cold", "hace frío"), ("it is hot", "hace calor"),
]
TEST = ["a red car", "the small window", "my sister reads", "it is hot"]


def write_tsv(pairs, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(pairs, columns=["src", "tgt"]).to_csv(path, sep="\t", index=False)
    return Path(path)


train_path = write_tsv(PAIRS * 12, "data/tutorial-3-train.tsv")
train_set = NMTDataset(train_path)

config = Config(d_model=64, n_heads=4, num_encoder_layers=2, num_decoder_layers=2, d_ff=128,
                dropout=0.1, batch_size=8, learning_rate=1e-3)
torch.manual_seed(0)
model = SimpleTransformer(src_vocab_size=len(train_set.src_vocab),
                          tgt_vocab_size=len(train_set.tgt_vocab), config=config).to(device)
loader = DataLoader(train_set, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=0)
# The checkpointer makes the run resumable, as in Lecture 7 and the A8 notebook.
train_model(model, loader, num_epochs=40, device=device, config=config, gradient_clip=1.0,
            log_every=0, checkpointer=TrainingCheckpointer(
                "tutorial-3", checkpoint_dir="checkpoints/tutorial-3", verbose=False))
model.eval()

for src, out in zip(TEST, translate_batch(model, TEST, train_set.src_vocab,
                                          train_set.tgt_vocab, device=device)):
    print(f"{src:18} -> {out}")

Epoch 1/40 | Train: 4.3933
Epoch 2/40 | Train: 4.2899


Epoch 3/40 | Train: 4.0965
Epoch 4/40 | Train: 3.9288


Epoch 5/40 | Train: 3.6615
Epoch 6/40 | Train: 3.4016


Epoch 7/40 | Train: 3.2607
Epoch 8/40 | Train: 3.0808


Epoch 9/40 | Train: 2.9528
Epoch 10/40 | Train: 2.8093


...

Epoch 31/40 | Train: 0.8134
Epoch 32/40 | Train: 0.8038


Epoch 33/40 | Train: 0.7888
Epoch 34/40 | Train: 0.7740


Epoch 35/40 | Train: 0.7680
Epoch 36/40 | Train: 0.7601


Epoch 37/40 | Train: 0.7568
Epoch 38/40 | Train: 0.7530


Epoch 39/40 | Train: 0.7450
Epoch 40/40 | Train: 0.7467
a red car          -> un coche rojo
the small window   -> la ventana pequeña
my sister reads    -> mi hermana lee
it is hot          -> hace calor


## Part 2: What the file holds

`save_checkpoint` writes one file with three things in it: the **weights**, the **model
config** (every size needed to build an empty model of the same shape), and a format tag. It
can also hold a SentencePiece tokenizer, and does when you give it one: a subword model is a
file, so it travels with the weights. A word vocabulary like this one is not; it is rebuilt
from the training data each time.

Look at the shape of the first layer. It has one row per word in the source vocabulary, and
nothing in the file says which word each row is.

In [5]:
model_config = dict(
    src_vocab_size=len(train_set.src_vocab), tgt_vocab_size=len(train_set.tgt_vocab),
    d_model=64, n_heads=4, num_encoder_layers=2, num_decoder_layers=2, d_ff=128,
)
save_checkpoint("checkpoints/tutorial-3/model.pt", model, model_config=model_config)

saved = load_checkpoint("checkpoints/tutorial-3/model.pt", map_location=device)
weights = saved["model_state_dict"]
print("in the file:", list(saved))
print("model_config:", saved["model_config"])
embedding = weights["src_tok_emb.weight"]
print(f"\nsrc_tok_emb.weight: {tuple(embedding.shape)}  "
      f"-> {embedding.shape[0]} rows, one per source word; which word is not recorded")

in the file: ['format', 'model_state_dict', 'model_config']
model_config: {'src_vocab_size': 39, 'tgt_vocab_size': 37, 'd_model': 64, 'n_heads': 4, 'num_encoder_layers': 2, 'num_decoder_layers': 2, 'd_ff': 128}

src_tok_emb.weight: (39, 64)  -> 39 rows, one per source word; which word is not recorded


## Part 3: Rebuild it right

A fresh session has none of the objects above. It rebuilds three things, in this order: the
vocabularies, from the **same training file**; an empty model, from the saved config; and the
weights, poured into that model. The helper below does exactly that, from whatever training
file it is given, and checks the one thing the config can check: that the vocabulary sizes are
the sizes the model was built for.

In [6]:
def rebuild(training_file):
    """Rebuild the vocabularies from training_file, then the saved model, and translate TEST."""
    vocab = NMTDataset(training_file)
    expected = (saved["model_config"]["src_vocab_size"], saved["model_config"]["tgt_vocab_size"])
    found = (len(vocab.src_vocab), len(vocab.tgt_vocab))
    print(f"vocabulary sizes: saved model expects {expected}, rebuilt {found}")
    rebuilt = SimpleTransformer(src_vocab_size=found[0], tgt_vocab_size=found[1],
                                config=config).to(device)
    rebuilt.load_state_dict(saved["model_state_dict"])
    rebuilt.eval()
    for src, out in zip(TEST, translate_batch(rebuilt, TEST, vocab.src_vocab,
                                              vocab.tgt_vocab, device=device)):
        print(f"  {src:18} -> {out}")


rebuild(train_path)

vocabulary sizes: saved model expects (39, 37), rebuilt (39, 37)
  a red car          -> un coche rojo
  the small window   -> la ventana pequeña
  my sister reads    -> mi hermana lee
  it is hot          -> hace calor


## Part 4: One extra word, and it stops you

Now rebuild from a training file with one more phrase in it, "a purple car": the kind of change
that happens when a split is redone or a cleaning step is rerun. Two new words, so each
vocabulary is one row longer than the weights. The model's layers no longer have the shape of
the saved ones, and loading refuses.

In [7]:
one_more = write_tsv((PAIRS + [("a purple car", "un coche morado")]) * 12,
                     "data/tutorial-3-one-more.tsv")
try:
    rebuild(one_more)
except RuntimeError as error:
    mismatch = next(line for line in str(error).splitlines() if "size mismatch" in line)
    print(f"RuntimeError:{mismatch}")

vocabulary sizes: saved model expects (39, 37), rebuilt (40, 38)
RuntimeError:	size mismatch for src_tok_emb.weight: copying a param with shape torch.Size([39, 64]) from checkpoint, the shape in current model is torch.Size([40, 64]).


## Part 5: The same words in another order, and nothing stops you

This time the training file holds exactly the same sentences, shuffled, as a split redone with a
different random seed would be. The vocabularies come out the same size, so the model loads
cleanly. But a vocabulary numbers words in the order it first meets them, so every number now
means a different word. Read the output: Spanish words, from the right vocabulary, in nonsense.
No error, no warning. Here you can see it is wrong; on 2,000 test sentences of your own
language you would see a low BLEU and blame the model.

In [8]:
rows = PAIRS * 12
random.Random(1).shuffle(rows)
reordered = write_tsv(rows, "data/tutorial-3-reordered.tsv")

rebuild(reordered)

vocabulary sizes: saved model expects (39, 37), rebuilt (39, 37)
  a red car          -> rojo mi hermana
  the small window   -> pequeña casa nueva
  my sister reads    -> dos canta ventana
  it is hot          -> hace calor un


## What to take away

- **A checkpoint is half a model.** It holds the weights and the config. A word vocabulary is
  not in it, so rebuild that from the *same file, in the same order* the model trained on.
- **A size check catches only the loud failure.** Comparing the rebuilt vocabulary sizes with
  the saved config, as `rebuild` prints, names the extra-word case before loading even fails.
  Nothing in the file can catch a reordering.
- **The cure is to save the vocabulary with the weights.** A SentencePiece model is a file, and
  `save_checkpoint` stores it inside the checkpoint, so the numbering cannot drift. That is one
  reason, beyond unknown words, to move from words to subwords.